<center><h1>Last_First_HW3</h1></center>
<br>
<br>

Name:
Yao Lin

Github Username:
ylin9754

USC ID:
8715431789

## 1. Time Series Classification Part 1: Feature Creation/Extraction

### (a) Download Data

Package imports

In [4]:
import os
import zipfile
import pandas as pd
import numpy as np

Get the AReM Data Set

In [10]:
zip_file = 'Homework 3, 4 Data.zip'

with zipfile.ZipFile(zip_file, 'r') as zip_ref:
    zip_ref.extractall('/content')

base_path = '/content/AReM'

os.listdir(base_path)

['walking',
 'bending2',
 'sitting',
 'sensorsPlacement.pdf',
 'standing',
 'cycling',
 'bending1',
 'bendingType.pdf',
 'lying']

### (b) Test and Train Data

In [6]:
train_files = []
test_files = []

folders = [
    'bending1',
    'bending2',
    'cycling',
    'lying',
    'sitting',
    'standing',
    'walking'
]

for folder in folders:
    folder_path = os.path.join(base_path, folder)
    files = [file for file in os.listdir(folder_path) if file.endswith('.csv')]

    for file in files:
        file_path = os.path.join(folder_path, file)

        if folder == 'bending1' or folder == 'bending2':
            if file == 'dataset1.csv' or file == 'dataset2.csv':
                test_files.append(file_path)
            else:
                train_files.append(file_path)

        else:
            if file == 'dataset1.csv' or file == 'dataset2.csv' or file == 'dataset3.csv':
                test_files.append(file_path)
            else:
                train_files.append(file_path)

print("Training files:", len(train_files))
print("Test files:", len(test_files))
print("Total files:", len(train_files) + len(test_files))

Training files: 69
Test files: 19
Total files: 88


### (c) Feature Extraction

#### i. Research

Common time-domain features used in time series classification include:

- Minimum
- Maximum
- Mean
- Median
- Variance
- Standard deviation
- First quartile
- Third quartile
- Interquartile range
- Range
- Skewness
- Kurtosis
- Root mean square (RMS)
- Zero-crossing rate
- Autocorrelation

#### ii. Extraction

In [7]:
column_names = [
    'time',
    'avg_rss12',
    'var_rss12',
    'avg_rss13',
    'var_rss13',
    'avg_rss23',
    'var_rss23'
]

all_files = []

for folder in folders:
    folder_path = os.path.join(base_path, folder)

    files = [file for file in os.listdir(folder_path) if file.endswith('.csv')]
    files = sorted(files, key=lambda x: int(x.replace('dataset', '').replace('.csv', '')))

    for file in files:
        all_files.append((folder, file))

feature_rows = []

for folder, file in all_files:

    file_path = os.path.join(base_path, folder, file)

    if folder == 'bending2' and file == 'dataset4.csv':
        df = pd.read_csv(
            file_path,
            comment='#',
            header=None,
            names=column_names,
            sep=r'\s+'
        )
    else:
        df = pd.read_csv(
            file_path,
            comment='#',
            header=None,
            names=column_names,
            usecols=range(7)
        )

    row = []

    for column in column_names[1:]:
        row.append(df[column].min())
        row.append(df[column].max())
        row.append(df[column].mean())
        row.append(df[column].median())
        row.append(df[column].std())
        row.append(df[column].quantile(0.25))
        row.append(df[column].quantile(0.75))

    feature_rows.append(row)

feature_names = []

for i in range(1, 7):
    feature_names.extend([
        'min' + str(i),
        'max' + str(i),
        'mean' + str(i),
        'median' + str(i),
        'std' + str(i),
        '1st_quart' + str(i),
        '3rd_quart' + str(i)
    ])

features = pd.DataFrame(feature_rows, columns=feature_names)

features.insert(0, 'Instance', range(1, len(features) + 1))

features

,Instance,min1,max1,mean1,median1,std1,1st_quart1,3rd_quart1,min2,max2,...,std5,1st_quart5,3rd_quart5,min6,max6,mean6,median6,std6,1st_quart6,3rd_quart6
0,1,37.25,45.00,40.624792,40.50,1.476967,39.25,42.0000,0.0,1.30,...,2.188449,33.0000,36.00,0.00,1.92,0.570583,0.43,0.582915,0.0000,1.3000
1,2,38.00,45.67,42.812812,42.50,1.435550,42.00,43.6700,0.0,1.22,...,1.995255,32.0000,34.50,0.00,3.11,0.571083,0.43,0.601010,0.0000,1.3000
2,3,35.00,47.40,43.954500,44.33,1.558835,43.00,45.0000,0.0,1.70,...,1.999604,35.3625,36.50,0.00,1.79,0.493292,0.43,0.513506,0.0000,0.9400
3,4,33.00,47.75,42.179812,43.50,3.670666,39.15,45.0000,0.0,3.00,...,3.849448,30.4575,36.33,0.00,2.18,0.613521,0.50,0.524317,0.0000,1.0000
4,5,33.00,45.75,41.678063,41.75,2.243490,41.33,42.7500,0.0,2.83,...,2.411026,28.4575,31.25,0.00,1.79,0.383292,0.43,0.389164,0.0000,0.5000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
83,84,19.50,45.33,33.586875,34.25,4.650935,30.25,37.0000,0.0,14.67,...,3.283983,13.7300,18.25,0.00,8.32,3.259729,3.11,1.640243,2.0500,4.3225
84,85,19.75,45.50,34.322750,35.25,4.752477,31.00,38.0000,0.0,13.47,...,3.119856,13.5000,17.75,0.00,9.67,3.432563,3.20,1.732727,2.1575,4.5650
85,86,19.50,46.00,34.546229,35.25,4.842294,31.25,37.8125,0.0,12.47,...,2.823124,14.0000,17.75,0.00,10.00,3.338125,3.08,1.656742,2.1600,4.3350
86,87,23.50,46.25,34.873229,35.25,4.531720,31.75,38.2500,0.0,14.82,...,3.131076,13.7500,18.00,0.00,9.51,3.424646,3.27,1.690960,2.1700,4.5000


#### iii. Standard Deviation

In [8]:
np.random.seed(1)

feature_std = []
lower_ci = []
upper_ci = []

for column in features.columns[1:]:

    values = features[column].values

    std_value = np.std(values, ddof=1)
    feature_std.append(std_value)

    bootstrap_std = []

    for i in range(1000):
        sample = np.random.choice(values, size=len(values), replace=True)
        bootstrap_std.append(np.std(sample, ddof=1))

    lower_ci.append(np.percentile(bootstrap_std, 5))
    upper_ci.append(np.percentile(bootstrap_std, 95))

std_table = pd.DataFrame({
    'Feature': features.columns[1:],
    'Standard Deviation': feature_std,
    '90% CI Lower': lower_ci,
    '90% CI Upper': upper_ci
})

std_table

,Feature,Standard Deviation,90% CI Lower,90% CI Upper
0,min1,9.569975,8.223994,10.737346
1,max1,4.394362,3.270589,5.307533
2,mean1,5.335718,4.676143,5.874374
3,median1,5.440054,4.804815,6.005083
4,std1,1.772153,1.591574,1.942295
5,1st_quart1,6.153590,5.575104,6.654749
6,3rd_quart1,5.138925,4.310069,5.824177
7,min2,0.000000,0.000000,0.000000
8,max2,5.062729,4.623785,5.424020
9,mean2,1.574164,1.413569,1.707852


#### iv. Select Features

I selected minimum, mean, and maximum as the three most important time-domain features.

The minimum and maximum capture the lower and upper extremes of each time series, while the mean represents its overall central level. Together, these three features provide a simple summary of the range and typical value of the signal.

## 2. ISLR 3.7.4

### (a) Linear Train

The training RSS for the cubic regression will be lower than or equal to the training RSS for the linear regression.

The cubic model is more flexible and includes the linear model as a special case when β2 = β3 = 0. Therefore, it can fit the training data at least as well as the linear model. Even though the true relationship is linear, the cubic model may also fit some random noise in the training data, which can make its training RSS slightly lower.

### (b) Linear Test

For test RSS, the linear regression is expected to perform better than the cubic regression because the true relationship is linear.

The cubic model is more flexible and may overfit the training data by fitting random noise. This can increase variance and lead to worse performance on new test observations. Therefore, we would generally expect the linear model to have a lower test RSS.

### (c) Not Linear Train

The training RSS for the cubic regression will be lower than or equal to the training RSS for the linear regression.

The cubic model is more flexible and includes the linear model as a special case. Since the true relationship is nonlinear, the extra quadratic and cubic terms may help the model capture some of that nonlinearity. Therefore, we would expect the cubic regression to fit the training data better and have a lower training RSS.

### (d) Not Linear Testing

There is not enough information to determine which model will have the lower test RSS.

If the true relationship is sufficiently nonlinear and is well approximated by a cubic function, the cubic regression may have a lower test RSS. However, if the relationship is only slightly nonlinear, the extra flexibility of the cubic model may increase variance and hurt test performance. Therefore, the better model depends on how nonlinear the true relationship actually is.

## 3. ISLR 3.7.3 - Extra Practice

## 3. ISLR 3.7.5 - Extra Practice

### AI Usage Disclosure

For Question 1(c)(iii), I used ChatGPT to help understand and implement the 90% bootstrap confidence interval for the standard deviation of each feature.

**Prompt Used:**  
"How do I build a 90% bootstrap confidence interval for the standard deviation of each feature?"

### Works Cited

“Homework 3 bootstrap confidence interval assistance” prompt. *ChatGPT*, OpenAI, 11 Oct. 2026.